# Normalizar nombres de equipos
Usa un diccionario para unificar nombres entre CSVs. Ajusta `TEAM_MAP` segun tus archivos.

In [ ]:
import pandas as pd
from pathlib import Path

# Update this map with exact keys as they appear in your CSVs.
TEAM_MAP = {
    # Example: raw_name: normalized_name
    # "Atle_Tucuman": "AtlTucuman",
    # "Cen._Cordoba–SdE": "CentralCordobaSdE",
    # "CF_Montreal": "CF Montreal",
}

# CSVs to normalize (edit paths)
input_paths = [
    r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat.csv",
    # r"C:\Users\ruben\Desktop\TEG\EstadiumsDB.csv",
    # r"C:\Users\ruben\Desktop\TEG\StadiumsEconomyDB_062251.csv",
    # r"C:\Users\ruben\Desktop\TEG\MacroEconomy2.csv",
]

output_dir = Path(r"C:\Users\ruben\Desktop\TEG")
output_dir.mkdir(parents=True, exist_ok=True)

def normalize_series(series: pd.Series) -> pd.Series:
    return series.replace(TEAM_MAP)

def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    team_cols = [c for c in ["team", "Team", "home_team", "away_team"] if c in df.columns]
    for col in team_cols:
        df[col] = normalize_series(df[col])
    return df

for path_str in input_paths:
    path = Path(path_str)
    if not path.exists():
        print(f"Skip (not found): {path}")
        continue
    df = pd.read_csv(path)
    df_norm = normalize_dataframe(df)
    out_path = output_dir / f"{path.stem}_normalized{path.suffix}"
    df_norm.to_csv(out_path, index=False)
    print(f"Wrote: {out_path}")

Wrote: C:\Users\ruben\Desktop\TEG\team_match_concat_normalized.csv


In [ ]:
from difflib import get_close_matches
import re

# Sources to build the review table
reference_path = r"C:\Users\ruben\Desktop\TEG\EstadiumsDB.csv"
sources_for_review = [
    r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat.csv",
    r"C:\Users\ruben\Desktop\TEG\StadiumsEconomyDB_062251.csv",
]

review_output = Path(r"C:\Users\ruben\Desktop\TEG\team_name_review.csv")

def collect_team_values(df: pd.DataFrame) -> set[str]:
    cols = [c for c in ["team", "Team", "home_team", "away_team"] if c in df.columns]
    values = set()
    for col in cols:
        values.update(df[col].dropna().astype(str).unique())
    return values

def normalize_key(name: str) -> str:
    # Lowercase, remove punctuation, normalize whitespace
    key = re.sub(r"[^\w]+", " ", name, flags=re.UNICODE).strip().lower()
    key = re.sub(r"\s+", " ", key)
    return key

# Explicit aliases for known variants
ALIAS_MAP = {
    # Real Oviedo variants
    "oviedo": "Real Oviedo",
    "real oviedo": "Real Oviedo",
    # Paris Saint-Germain variants
    "psg": "PSG",
    "paris sg": "PSG",
    "paris s g": "PSG",
    "paris saint germain": "PSG",
    "paris saint germain fc": "PSG",
    "paris s g fc": "PSG",
}

ref_df = pd.read_csv(reference_path)
ref_teams = sorted(ref_df["Team"].dropna().astype(str).unique())
ref_teams_set = set(ref_teams)

rows = []
for src_path in sources_for_review:
    df = pd.read_csv(src_path)
    for raw in sorted(collect_team_values(df)):
        key = normalize_key(raw)
        alias = ALIAS_MAP.get(key, "")

        # If alias exists but is not in reference, keep as suggestion anyway for review
        if alias:
            suggestion = alias
        else:
            suggestion_match = get_close_matches(raw, ref_teams, n=1, cutoff=0.6)
            suggestion = suggestion_match[0] if suggestion_match else ""

        rows.append({
            "source_file": Path(src_path).name,
            "raw_name": raw,
            "normalized_key": key,
            "suggested_normalized": suggestion,
            "approved_normalized": "",
        })

review_df = pd.DataFrame(rows)
review_df.to_csv(review_output, index=False)
print(f"Wrote review table: {review_output}")

Wrote review table: C:\Users\ruben\Desktop\TEG\team_name_review.csv


In [ ]:
# Manual overrides from your review
manual_overrides = {
    "AEK": "AEK",
    "AEK Athens": "AEK",
    "AEK_Athens": "AEK",
    "Al Suqoor Saudi Club": "AlSuqoorSaudiClub",
    "Al_Suqoor_Saudi_Club": "AlSuqoorSaudiClub",
    "Al-Najma": "AlNajma",
    "Atlletico Goianiense": "AtleticoGO",
    "AtlGoianiense": "AtleticoGO",
    "Atl Goianiense": "AtleticoGO",
    "Atl_Goaniense": "AtleticoGO",
    "Atl_Goianiense": "AtleticoGO",
    "Atlético": "AtlSanLuis",
    "AtlÃ©tico": "AtlSanLuis",
    "Bolivar_SC": "BolivarSC",
    "Bolívar SC": "BolivarSC",
    "Girondis": "GirondisBordeaux",
    "Bordeaux": "GirondisBordeaux",
    "CF Montréal": "Montreal",
    "CF Montreal": "Montreal",
    "CF_Montreal": "Montreal",
    "MontImpact": "Montreal",
    "Montreal_Impact": "Montreal",
    "Montreal Impact": "Montreal",
    "Düsseldorf": "Dusseldorf",
    "Dusseldorg": "Dusseldorf",
    "Dusseldorf": "Dusseldorf",
    "Fortuna95": "Dusseldorf",
    "Deportivo JBL": "DepJBL",
    "Deportivo_JBL": "DepJBL",
    "Hermanos Colmenarez": "InterBarinas",
    "Huracán": "Huracan",
    "Inter": "InterMilan",
    "Internazionale": "InterMilan",
    "InterPA": "InterPA",
    "Internacional": "InterPA",
    "Kheybar": "KheybarKhorramabad",
    "KheybarKhorramabad": "KheybarKhorramabad",
    "Larissa": "AEL",
    "AEL": "AEL",
    "Bayer04": "Leverkusen",
    "Leverkusen": "Leverkusen",
    "Nassaji Mazandaran": "NassajiMazandaran",
    "Nassaji_Mazandaran": "NassajiMazandaran",
    "Nassaji": "NassajiMazandaran",
    "NY Red Bulls": "NYRB",
    "NY_Red_Bulls": "NYRB",
    "NYRB": "NYRB",
    "OFI": "OFI",
    "OFI Crete": "OFI",
    "OFI_Crete": "OFI",
    "Omiya Ardija": "OmiyaArdija",
    "Omiya_Ardija": "OmiyaArdija",
    "Paraná": "Parana",
    "Parana": "Parana",
    "Paris_S-G": "PSG",
    "Paris Saint-Germain": "PSG",
    "Paris S-G": "PSG",
    "Paris_Saint-Germain": "PSG",
    "Paris FC": "ParisFC",
    "Paris_FC": "ParisFC",
    "PAS": "Giannina",
    "PAS Giannina": "Giannina",
    "PAS_Giannina": "Giannina",
    "Pisa": "Pisa",
    "Racing Club": "RacingAvellaneda",
    "Racing_Club": "RacingAvellaneda",
    "RacingAvellaneda": "RacingAvellaneda",
    "RacingLens": "RacingLens",
    "Lens": "RacingLens",
    "Rayo Zuliano": "Zulia",
    "Rayo_Zuliano": "Zulia",
    "Zulia": "Zulia",
    "Rennes": "StadeRennais",
    "StadeRennais": "StadeRennais",
    "San_Marino": "SanMarino",
    "SorkhPooshan FC": "SorkhPooshanFC",
    "SorkhPooshan_FC": "SorkhPooshanFC",
    "Sporting KC": "SpKansasCity",
    "Sporting_KC": "SpKansasCity",
    "UANL": "TigresUANL",
    "TigresUANL": "TigresUANL",
    "Unión": "UnionSF",
    "Union": "UnionSF",
    "UnionSF": "UnionSF",
    "Universidad Central": "UCVFC",
    "Universidad_Central": "UCVFC",
    "Volos": "Volos",
    "Volos N.F.C.": "Volos",
    "Volos NFC": "Volos",
    "Volos_N.F.C.": "Volos",
    "Volos_NFC": "Volos",
    "Vortis": "TokushimaVortis",
    "Tokushima Vortis": "TokushimaVortis",
    "TokushimaVortis": "TokushimaVortis",
    "Tokushima_Vortis": "TokushimaVortis",
    "Yamaga": "MatsumotoYamaga",
    "MatsumotoYamaga": "MatsumotoYamaga",
    "Athletic Club": "AthBilbao",
    "Athletic_Club": "AthBilbao",
    "AthBilbao": "AthBilbao",
}

# Apply overrides to the review table
review_path = Path(review_output)
if not review_path.exists():
    raise FileNotFoundError(f"Missing review file: {review_path}")

review_df = pd.read_csv(review_path)
review_df["approved_normalized"] = review_df.apply(
    lambda row: manual_overrides.get(row["raw_name"], row.get("approved_normalized", "")),
    axis=1,
)

# Ensure suggested_normalized exists even if a previous run removed it
if "suggested_normalized" not in review_df.columns:
    review_df["suggested_normalized"] = ""

# Merge suggested/approved into a single column with approved priority
approved_series = review_df["approved_normalized"].fillna("").astype(str).str.strip()
suggested_series = review_df["suggested_normalized"].fillna("").astype(str).str.strip()
review_df["normalized"] = approved_series.where(approved_series != "", suggested_series)

review_df.to_csv(review_path, index=False)
print(f"Updated review table: {review_path}")

# Build TEAM_MAP from normalized values
approved = review_df.dropna(subset=["normalized"])
approved = approved[approved["normalized"].astype(str).str.len() > 0]
team_map = dict(zip(approved["raw_name"], approved["normalized"]))
team_map_path = Path(r"C:\Users\ruben\Desktop\TEG\team_map_manual.csv")
pd.DataFrame({"raw_name": team_map.keys(), "normalized_name": team_map.values()}).to_csv(team_map_path, index=False)
print(f"Wrote TEAM_MAP CSV: {team_map_path}")
print(f"TEAM_MAP size: {len(team_map)}")

Updated review table: C:\Users\ruben\Desktop\TEG\team_name_review.csv
Wrote TEAM_MAP CSV: C:\Users\ruben\Desktop\TEG\team_map_manual.csv
TEAM_MAP size: 933


In [ ]:
# Base manual dictionary (your curated entries)
manual_dict_base = {
    "AEK": "AEK",
    "AEK Athens": "AEK",
    "AEK_Athens": "AEK",
    "Al Suqoor Saudi Club": "AlSuqoorSaudiClub",
    "Al_Suqoor_Saudi_Club": "AlSuqoorSaudiClub",
    "Al-Najma": "AlNajma",
    "Atlletico Goianiense": "AtleticoGO",
    "AtlGoianiense": "AtleticoGO",
    "Atl Goianiense": "AtleticoGO",
    "Atl_Goaniense": "AtleticoGO",
    "Atl_Goianiense": "AtleticoGO",
    "Atlético": "AtlSanLuis",
    "AtlÃ©tico": "AtlSanLuis",
    "Bolivar_SC": "BolivarSC",
    "Bolívar SC": "BolivarSC",
    "Girondis": "GirondisBordeaux",
    "Bordeaux": "GirondisBordeaux",
    "CF Montréal": "Montreal",
    "CF Montreal": "Montreal",
    "CF_Montreal": "Montreal",
    "MontImpact": "Montreal",
    "Montreal_Impact": "Montreal",
    "Montreal Impact": "Montreal",
    "Düsseldorf": "Dusseldorf",
    "Dusseldorg": "Dusseldorf",
    "Dusseldorf": "Dusseldorf",
    "Fortuna95": "Dusseldorf",
    "Deportivo JBL": "DepJBL",
    "Deportivo_JBL": "DepJBL",
    "Hermanos Colmenarez": "InterBarinas",
    "Huracán": "Huracan",
    "Huracan": "Huracan",
    "Inter": "InterMilan",
    "Internazionale": "InterMilan",
    "InterPA": "InterPA",
    "Internacional": "InterPA",
    "Kheybar": "KheybarKhorramabad",
    "KheybarKhorramabad": "KheybarKhorramabad",
    "Larissa": "AEL",
    "AEL": "AEL",
    "Bayer04": "Leverkusen",
    "Leverkusen": "Leverkusen",
    "Nassaji Mazandaran": "NassajiMazandaran",
    "Nassaji_Mazandaran": "NassajiMazandaran",
    "Nassaji": "NassajiMazandaran",
    "NassajiManzandaran": "MassajiMazandaran",
    "NY Red Bulls": "NYRB",
    "NY_Red_Bulls": "NYRB",
    "NYRB": "NYRB",
    "OFI": "OFI",
    "OFI Crete": "OFI",
    "OFI_Crete": "OFI",
    "Omiya Ardija": "OmiyaArdija",
    "Omiya_Ardija": "OmiyaArdija",
    "Paraná": "Parana",
    "Parana": "Parana",
    "Paris_S-G": "PSG",
    "Paris Saint-Germain": "PSG",
    "Paris S-G": "PSG",
    "Paris_Saint-Germain": "PSG",
    "Paris FC": "ParisFC",
    "Paris_FC": "ParisFC",
    "PAS": "Giannina",
    "PAS Giannina": "Giannina",
    "PAS_Giannina": "Giannina",
    "Pisa": "Pisa",
    "Racing Club": "RacingAvellaneda",
    "Racing_Club": "RacingAvellaneda",
    "RacingAvellaneda": "RacingAvellaneda",
    "RacingLens": "RacingLens",
    "Lens": "RacingLens",
    "Rayo Zuliano": "Zulia",
    "Rayo_Zuliano": "Zulia",
    "Zulia": "Zulia",
    "Rennes": "StadeRennais",
    "StadeRennais": "StadeRennais",
    "San_Marino": "SanMarino",
    "SorkhPooshan FC": "SorkhPooshanFC",
    "SorkhPooshan_FC": "SorkhPooshanFC",
    "Sporting KC": "SpKansasCity",
    "Sporting_KC": "SpKansasCity",
    "UANL": "TigresUANL",
    "TigresUANL": "TigresUANL",
    "Unión": "UnionSF",
    "Union": "UnionSF",
    "UnionSF": "UnionSF",
    "Universidad Central": "UCVFC",
    "Universidad_Central": "UCVFC",
    "Volos": "Volos",
    "Volos N.F.C.": "Volos",
    "Volos NFC": "Volos",
    "Volos_N.F.C.": "Volos",
    "Volos_NFC": "Volos",
    "Vortis": "TokushimaVortis",
    "Tokushima Vortis": "TokushimaVortis",
    "TokushimaVortis": "TokushimaVortis",
    "Tokushima_Vortis": "TokushimaVortis",
    "Yamaga": "MatsumotoYamaga",
    "MatsumotoYamaga": "MatsumotoYamaga",
    "Athletic Club": "AthBilbao",
    "Athletic_Club": "AthBilbao",
    "AthBilbao": "AthBilbao",
    "Pohang": "PohangSteelers",
    "Deportivo Lara": "DepLara",
    "Deportivo_Lara": "DepLara",
    "Morelia": "Mazatlan",
    "Atletico_Goianiense": "AtleticoGO",
    "Atletico_Socopo": "AtlSocopo",
}

# Merge with CSV suggestions without overwriting existing keys
review_df = pd.read_csv(review_output)
approved_series = review_df.get("approved_normalized", pd.Series(dtype=str)).fillna("").astype(str).str.strip()
suggested_series = review_df.get("suggested_normalized", pd.Series(dtype=str)).fillna("").astype(str).str.strip()
normalized = approved_series.where(approved_series != "", suggested_series)

csv_entries = dict(zip(review_df["raw_name"], normalized))
csv_entries = {k: v for k, v in csv_entries.items() if isinstance(v, str) and v.strip()}

manual_dict = dict(manual_dict_base)
for key, value in csv_entries.items():
    if key not in manual_dict:
        manual_dict[key] = value

print(f"manual_dict size: {len(manual_dict)}")
print(f"Added from CSV: {len(manual_dict) - len(manual_dict_base)}")

manual_dict size: 940
Added from CSV: 832


In [ ]:
dict(sorted(manual_dict.items()))

{'AC Milan': 'Milan',
 'AC_Milan': 'Milan',
 'AEK': 'AEK',
 'AEK Athens': 'AEK',
 'AEK_Athens': 'AEK',
 'AEL': 'AEL',
 'AO Kifisia': 'Kifisia',
 'AO_Kifisia': 'Kifisia',
 'AS Roma': 'Roma',
 'AS_Roma': 'Roma',
 'Abha': 'Abha',
 'AcPtoCabello': 'AcPtoCabello',
 'Acad Pr Cabello': 'AcPtoCabello',
 'Acad_Pr_Cabello': 'AcPtoCabello',
 'Ajaccio': 'Ajaccio',
 'Al Kholood Saudi Club': 'AlKholood',
 'Al Okhdood SC': 'AlOkhdood',
 'Al Suqoor Saudi Club': 'AlSuqoorSaudiClub',
 'Al-Adalah': 'AlAdalah',
 'Al-Ahli': 'AlAhli',
 'Al-Ain': 'AlAin',
 'Al-Batin': 'AlBatin',
 'Al-Ettifaq': 'AlEttifaq',
 'Al-Faisaly': 'AlFaisaly',
 'Al-Fateh': 'AlFateh',
 'Al-Fayha': 'AlFayha',
 'Al-Hazem': 'AlHazem',
 'Al-Hilal': 'AlHilal',
 'Al-Ittihad': 'AlIttihad',
 'Al-Khaleej': 'AlKhaleej',
 'Al-Najma': 'AlNajma',
 'Al-Nassr': 'AlNassr',
 'Al-Orobah': 'AlOrobah',
 'Al-Qadsiah': 'AlQadsiah',
 'Al-Raed': 'AlRaed',
 'Al-Riyadh SC': 'AlRiyadh',
 'Al-Riyadh_SC': 'AlRiyadh',
 'Al-Shabab': 'AlShabab',
 "Al-Ta'ee": 'AlTai',

In [ ]:
# Group variations by normalized team name
grouped = {}
for raw_name, norm_name in manual_dict.items():
    grouped.setdefault(norm_name, []).append(raw_name)

# Sort for readability
for norm_name in grouped:
    grouped[norm_name] = sorted(grouped[norm_name])

print(f"Total normalized teams: {len(grouped)}")

# Show all teams with their variations
for norm_name in sorted(grouped):
    print(f"{norm_name}: {', '.join(grouped[norm_name])}")

Total normalized teams: 435
AEK: AEK, AEK Athens, AEK_Athens
AEL: AEL, Larissa
Abha: Abha
AcPtoCabello: AcPtoCabello, Acad Pr Cabello, Acad_Pr_Cabello, Puerto Cabello, Puerto_Cabello
Ajaccio: Ajaccio
AlAdalah: Al-Adalah, AlAdalah
AlAhli: Al-Ahli, AlAhli
AlAin: Al-Ain, AlAin
AlBatin: Al-Batin, AlBatin
AlEttifaq: Al-Ettifaq, AlEttifaq
AlFaisaly: Al-Faisaly, AlFaisaly
AlFateh: Al-Fateh, AlFateh
AlFayha: Al-Fayha, AlFayha
AlHazem: Al-Hazem, AlHazem
AlHilal: Al-Hilal, AlHilal
AlIttihad: Al-Ittihad, AlIttihad
AlKhaleej: Al-Khaleej, AlKhaleej
AlKholood: Al Kholood Saudi Club, AlKholood, Al_Kholood_Saudi_Club
AlNajma: Al-Najma
AlNassr: Al-Nassr, AlNassr
AlOkhdood: Al Okhdood SC, AlOkhdood, Al_Okhdood_SC
AlOrobah: Al-Orobah, AlOrobah
AlQadsiah: Al-Qadsiah, AlQadsiah
AlRaed: Al-Raed, AlRaed
AlRiyadh: Al-Riyadh SC, Al-Riyadh_SC, AlRiyadh
AlShabab: Al-Shabab, AlShabab
AlSuqoorSaudiClub: Al Suqoor Saudi Club, Al_Suqoor_Saudi_Club
AlTaawoun: Al-Taawoun, AlTaawoun
AlTai: Al-Ta'ee, AlTai
AlWehda: Al-W

In [ ]:
# Flag items that need correction (automatic matching)
reference_path = r"C:\Users\ruben\Desktop\TEG\EstadiumsDB.csv"
ref_df = pd.read_csv(reference_path)
ref_teams = ref_df["Team"].dropna().astype(str).tolist()

def _norm_key(name: str) -> str:
    name = name.lower()
    name = "".join(ch if ch.isalnum() else " " for ch in name)
    return " ".join(name.split())

ref_key_map = {_norm_key(t): t for t in ref_teams}

missing_normalized = []
missing_in_reference = []

for raw_name, norm_name in manual_dict.items():
    if not isinstance(norm_name, str) or not norm_name.strip():
        missing_normalized.append(raw_name)
        continue
    norm_key = _norm_key(norm_name)
    if norm_key not in ref_key_map:
        missing_in_reference.append((raw_name, norm_name))

print(f"Missing normalized (count): {len(missing_normalized)}")
for raw_name in sorted(missing_normalized):
    print(f"MISSING_NORMALIZED: {raw_name}")

print(f"Missing in reference (count): {len(missing_in_reference)}")
for raw_name, norm_name in sorted(missing_in_reference, key=lambda x: x[1]):
    print(f"MISSING_IN_REFERENCE: {raw_name} -> {norm_name}")

Missing normalized (count): 0
Missing in reference (count): 49
MISSING_IN_REFERENCE: Al-Najma -> AlNajma
MISSING_IN_REFERENCE: Al Suqoor Saudi Club -> AlSuqoorSaudiClub
MISSING_IN_REFERENCE: Al_Suqoor_Saudi_Club -> AlSuqoorSaudiClub
MISSING_IN_REFERENCE: Atletico_Socopo -> AtlSocopo
MISSING_IN_REFERENCE: Bolivar_SC -> BolivarSC
MISSING_IN_REFERENCE: Bolívar SC -> BolivarSC
MISSING_IN_REFERENCE: Deportivo JBL -> DepJBL
MISSING_IN_REFERENCE: Deportivo_JBL -> DepJBL
MISSING_IN_REFERENCE: Düsseldorf -> Dusseldorf
MISSING_IN_REFERENCE: Dusseldorg -> Dusseldorf
MISSING_IN_REFERENCE: Dusseldorf -> Dusseldorf
MISSING_IN_REFERENCE: Fortuna95 -> Dusseldorf
MISSING_IN_REFERENCE: PAS -> Giannina
MISSING_IN_REFERENCE: PAS Giannina -> Giannina
MISSING_IN_REFERENCE: PAS_Giannina -> Giannina
MISSING_IN_REFERENCE: Girondis -> GirondisBordeaux
MISSING_IN_REFERENCE: Bordeaux -> GirondisBordeaux
MISSING_IN_REFERENCE: Huracán -> Huracan
MISSING_IN_REFERENCE: Huracan -> Huracan
MISSING_IN_REFERENCE: Inter -

In [ ]:
# Convert match_date (dd/mm/yyyy or dd-mm-yyyy) to mm/dd/yyyy and save a new CSV
input_path = r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat.csv"
input_path = Path(input_path)
output_path = input_path.with_name(f"{input_path.stem}_todatetime{input_path.suffix}")

df = pd.read_csv(input_path)
if "match_date" not in df.columns:
    raise KeyError("Column 'match_date' not found in CSV")

parsed = pd.to_datetime(df["match_date"], dayfirst=True, errors="coerce")
invalid_count = parsed.isna().sum()

df = df.copy()
df["match_date"] = parsed.dt.strftime("%m/%d/%Y")
df.loc[parsed.isna(), "match_date"] = ""

df.to_csv(output_path, index=False)
print(f"Wrote: {output_path}")
print(f"Invalid dates coerced to empty: {invalid_count}")

Wrote: C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat_todatetime.csv
Invalid dates coerced to empty: 0


In [ ]:
# Create a small sample CSV (first 10 rows) for quick tests
from pathlib import Path
import pandas as pd

concat_path = Path(r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat.csv")
sample_path = concat_path.with_name(f"{concat_path.stem}_sample20{concat_path.suffix}")

df_concat = pd.read_csv(concat_path)
df_concat.head(20).to_csv(sample_path, index=False)
print(f"Wrote sample: {sample_path}")

Wrote sample: C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat_sample20.csv


In [ ]:
# Quick integration test using the sample CSV (first 5 matches)
from MasterG2Av1226 import MatchAnalyst
from G2Av1226 import Glicko2, Rating

sample_path = Path(r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat_sample20.csv")
stadiums_path = Path(r"C:\Users\ruben\Desktop\TEG\EstadiumsDB.csv")
economy_path = Path(r"C:\Users\ruben\Desktop\TEG\StadiumsEconomyDB_062251.csv")

df_sports = pd.read_csv(sample_path)
df_stadiums = pd.read_csv(stadiums_path)
df_economy = pd.read_csv(economy_path)
df_macro = pd.DataFrame()

# Apply the normalization dictionary to sports and stadium teams (in-memory only).
if "manual_dict" in globals():
    df_sports = df_sports.copy()
    for col in ["team", "home_team", "away_team"]:
        if col in df_sports.columns:
            df_sports[col] = df_sports[col].replace(manual_dict)
    df_stadiums = df_stadiums.copy()
    df_stadiums["Team"] = df_stadiums["Team"].replace(manual_dict)

# Show which sample team names do not match the stadiums list exactly.
valid_teams = set(df_stadiums["Team"].dropna().astype(str))
sample_teams = set(df_sports["home_team"].dropna().astype(str)) | set(df_sports["away_team"].dropna().astype(str))
missing = sorted(sample_teams - valid_teams)
print(f"Missing teams in stadiums (count): {len(missing)}")
for name in missing:
    print(name)

analyst = MatchAnalyst(df_stadiums, df_economy, df_macro, df_sports=df_sports)
engine = Glicko2()

date_col = "match_date" if "match_date" in df_sports.columns else "date"
match_key = "match_id" if "match_id" in df_sports.columns else None

if match_key:
    rows = df_sports.dropna(subset=[match_key]).drop_duplicates(subset=[match_key]).head(5)
else:
    dedup_cols = [c for c in ["home_team", "away_team", date_col] if c in df_sports.columns]
    rows = df_sports.drop_duplicates(subset=dedup_cols).head(5)

for idx, row in rows.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    metrics = analyst.get_metrics(row[date_col], home, away)

    # Default ratings and neutral params for a quick probability check.
    probs = engine.probabilities(Rating(), Rating(), advantage=0.0, skellam_draw_probability=0.0)
    print(f"{home} vs {away}")
    print(f"metrics: {metrics}")
    print(f"probabilities (H/D/A): {probs}")
    print("-")

Missing teams in stadiums (count): 11
CF_Montreal
Jubilo_Iwata
Mallorca
Naft_Masjed_Soleyman
Philadelphia_Union
RB_Leipzig
Sanat_Mes_Kerman
Santos_Laguna
Shimizu_S-Pulse
UANL
Volos_N.F.C.
Malavan vs Naft_Masjed_Soleyman
metrics: {'xg': 1.35, 'or': np.float64(0.671875), 'po': np.float64(0.050321827378866146), 'd_hum': 0.0, 'd_tem': 0.0, 'd_mv': np.float64(nan)}
probabilities (H/D/A): (0.3547696064649047, 0.2904607870701907, 0.3547696064649047)
-
CF_Montreal vs Philadelphia_Union
metrics: {'xg': 1.35, 'or': 0.5, 'po': 0, 'd_hum': 0.0, 'd_tem': 0.0, 'd_mv': 0.0}
probabilities (H/D/A): (0.3547696064649047, 0.2904607870701907, 0.3547696064649047)
-
Alaves vs Cadiz
metrics: {'xg': 1.35, 'or': np.float64(0.8765625), 'po': np.float64(12.926233814447272), 'd_hum': 0.0, 'd_tem': 0.0, 'd_mv': np.float64(5.599999999999994)}
probabilities (H/D/A): (0.3547696064649047, 0.2904607870701907, 0.3547696064649047)
-
Jubilo_Iwata vs Shimizu_S-Pulse
metrics: {'xg': 1.35, 'or': 0.5, 'po': 0, 'd_hum': 0.0, 'd

In [ ]:
# Filter matches in the sample to teams present in EstadiumsDB
from pathlib import Path
import pandas as pd

sample_path = Path(r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat_sample20.csv")
stadiums_path = Path(r"C:\Users\ruben\Desktop\TEG\EstadiumsDB.csv")
output_path = sample_path.with_name("team_match_concat_sample20.csv")

df_concat = pd.read_csv(sample_path)
df_stadiums = pd.read_csv(stadiums_path)

# Apply the normalization dictionary to sports and stadium teams (in-memory only).
if "manual_dict" in globals():
    df_concat = df_concat.copy()
    for col in ["team", "home_team", "away_team"]:
        if col in df_concat.columns:
            df_concat[col] = df_concat[col].replace(manual_dict)
    df_stadiums = df_stadiums.copy()
    df_stadiums["Team"] = df_stadiums["Team"].replace(manual_dict)

valid_teams = set(df_stadiums["Team"].dropna().astype(str))

if not {"home_team", "away_team"}.issubset(df_concat.columns):
    raise KeyError("Expected columns 'home_team' and 'away_team' in sample CSV")

mask = df_concat["home_team"].isin(valid_teams) & df_concat["away_team"].isin(valid_teams)
df_filtered = df_concat[mask].copy()

df_filtered.to_csv(output_path, index=False)
print(f"Wrote filtered sample: {output_path}")
print(f"Rows before: {len(df_concat)} | after: {len(df_filtered)}")

['CF_Montreal', 'Jubilo_Iwata', 'Mallorca', 'Naft_Masjed_Soleyman', 'Philadelphia_Union', 'RB_Leipzig', 'Sanat_Mes_Kerman', 'Santos_Laguna', 'Shimizu_S-Pulse', 'UANL', 'Volos_N.F.C.']
Wrote filtered sample: C:\Users\ruben\Documents\Universidad\TEG\Codigos\VSCode\team_match_concat_sample20.csv
Rows before: 20 | after: 4
